# Parameter-efficient Finetuning with LoRA

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # For OpenAI's pretrained weights
        "pandas"      # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.11.2
numpy version: 2.5.1
tiktoken version: 0.14.0
torch version: 2.14.0
tensorflow version: 2.22.0rc0
pandas version: 3.0.5


## 1. Introduction to LoRA
全参数微调时的权重更新 $\Delta W$, 被约束为低秩 $\boldsymbol{\Delta W} \approx BA$；实现这个低秩更新，只需要少量可训练参数（即矩阵 $A,B$ 中的参数）
> LoRA 的更多理论知识，请阅读 `./LoRA.md`

## 2. Preparing the dataset

In [ ]:
# import urllib
import requests
from pathlib import Path
import pandas as pd
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

# The book originally used
# except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
# in the code above.
# However, some VPN users reported issues with `urllib`, so the code was updated
# to use `requests` instead

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)